## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

## Load Cleaned Dataset

In [2]:
# Load cleaned dataset
df = pd.read_csv('../data/data_cleaned.csv')
df_original = df.copy()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Dataset shape: (45194, 15)
Columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


## Step 1: Create Derived Features

In [ ]:
print("=== CREATING DERIVED FEATURES ===")

# 1. Education Rank Scores (based on education hierarchy)
education_rank = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4,
    '9th': 5, '10th': 6, '11th': 7, '12th': 8,
    'HS-grad': 9, 'Some-college': 10, 'Assoc-voc': 11, 'Assoc-acdm': 12,
    'Bachelors': 13, 'Masters': 14, 'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_rank)
df['education_rank'] = df['education_rank'].fillna(df['education-num'])  # Fallback

print("✓ Education rank scores created")

# 2. Age Groups
df['age_group'] = pd.cut(df['age'], 
                        bins=[0, 25, 35, 45, 55, 100], 
                        labels=['Young', 'Adult', 'Middle', 'Senior', 'Elder'])

print("✓ Age groups created")

# 3. Work Intensity (overtime indicator)
df['overtime'] = (df['hours-per-week'] > 40).astype(int)
df['work_intensity'] = pd.cut(df['hours-per-week'], 
                             bins=[0, 20, 40, 50, 100], 
                             labels=['Part-time', 'Standard', 'Over-time', 'Extreme'])

print("✓ Work intensity features created")

# 4. Capital Features (has capital gain/loss)
df['has_capital_gain'] = (df['capital-gain'] > 0).astype(int)
df['has_capital_loss'] = (df['capital-loss'] > 0).astype(int)
df['capital_net'] = df['capital-gain'] - df['capital-loss']

print("✓ Capital features created")

# 5. Education Groups
education_groups = {
    'Low': ['Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th', '11th', '12th'],
    'Medium': ['HS-grad', 'Some-college', 'Assoc-voc', 'Assoc-acdm'], 
    'High': ['Bachelors', 'Masters', 'Prof-school', 'Doctorate']
}

def map_education_group(education):
    for group, educations in education_groups.items():
        if education in educations:
            return group
    return 'Other'

df['education_group'] = df['education'].apply(map_education_group)

print("✓ Education groups created")

# 6. Marriage Status Simplified
married_status = ['Married-civ-spouse', 'Married-spouse-absent', 'Married-AF-spouse']
df['is_married'] = df['marital-status'].isin(married_status).astype(int)

print("✓ Marriage status feature created")

# 7. Native Country Grouped (US vs Other)
df['is_us_native'] = (df['native-country'] == 'United-States').astype(int)

print("✓ Native country feature created")

print(f"\nNew features created. Dataset shape: {df.shape}")
new_features = ['education_rank', 'age_group', 'overtime', 'work_intensity', 
                'has_capital_gain', 'has_capital_loss', 'capital_net', 
                'education_group', 'is_married', 'is_us_native']
print(f"New features: {new_features}")

## Step 2: Create Interaction Features

In [ ]:
print("=== CREATING INTERACTION FEATURES ===")

# 1. Age × Education interaction (experience with education level)
df['age_education_interaction'] = df['age'] * df['education_rank']

# 2. Hours × Education interaction (work commitment with education)
df['hours_education_interaction'] = df['hours-per-week'] * df['education_rank']

# 3. Age × Hours interaction (experience with work commitment)
df['age_hours_interaction'] = df['age'] * df['hours-per-week']

# 4. Marriage × Education interaction
df['marriage_education_interaction'] = df['is_married'] * df['education_rank']

print("✓ Interaction features created")
interaction_features = ['age_education_interaction', 'hours_education_interaction', 
                       'age_hours_interaction', 'marriage_education_interaction']
print(f"Interaction features: {interaction_features}")

print(f"Dataset shape after interactions: {df.shape}")

## Step 3: Encode Categorical Variables

In [ ]:
print("=== ENCODING CATEGORICAL VARIABLES ===")

# Identify categorical columns (exclude target)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_to_encode = [col for col in categorical_cols if col != 'income']

print(f"Categorical columns to encode: {categorical_to_encode}")

# Label Encoding for binary/ordinal columns
binary_cols = ['sex']  # Binary categories
label_encoders = {}

print("\nLabel encoding binary columns:")
for col in binary_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col + '_encoded'] = le.fit_transform(df[col])
        label_encoders[col] = le
        print(f"  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# One-Hot Encoding for nominal categorical columns
nominal_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 
                'race', 'native-country', 'education', 'age_group', 'work_intensity', 
                'education_group']
nominal_cols = [col for col in nominal_cols if col in df.columns]

print(f"\nOne-hot encoding nominal columns: {nominal_cols}")

# Create one-hot encoded features
df_encoded = pd.get_dummies(df, columns=nominal_cols, prefix_sep='_', drop_first=True, dtype=int)

print(f"Dataset shape after encoding: {df_encoded.shape}")
print(f"Columns expanded from {df.shape[1]} to {df_encoded.shape[1]}")

# Show sample of new columns
new_columns = [col for col in df_encoded.columns if col not in df.columns]
print(f"\nSample of new encoded columns: {new_columns[:10]}")

## Step 4: Prepare Target Variable

In [ ]:
print("=== PREPARING TARGET VARIABLE ===")

# Encode target variable
target_encoder = LabelEncoder()
df_encoded['income_target'] = target_encoder.fit_transform(df_encoded['income'])

# Show encoding mapping
target_mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))
print(f"Target encoding: {target_mapping}")

# Verify target distribution
target_dist = df_encoded['income_target'].value_counts()
print(f"\nTarget distribution:")
for value, count in target_dist.items():
    label = target_encoder.inverse_transform([value])[0]
    percentage = (count / len(df_encoded)) * 100
    print(f"  {label} ({value}): {count:,} ({percentage:.1f}%)")

## Step 5: Scale Numerical Features

In [ ]:
print("=== SCALING NUMERICAL FEATURES ===")

# Identify numerical columns to scale (exclude target and binary encoded)
numerical_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 
                     'hours-per-week', 'education_rank', 'capital_net'] + interaction_features

# Filter existing columns
numerical_features = [col for col in numerical_features if col in df_encoded.columns]
print(f"Numerical features to scale: {numerical_features}")

# Initialize scaler
scaler = StandardScaler()

# Create scaled dataset
df_scaled = df_encoded.copy()
df_scaled[numerical_features] = scaler.fit_transform(df_scaled[numerical_features])

print(f"\n✓ Numerical features scaled using StandardScaler")
print(f"Features scaled: {len(numerical_features)}")

# Show scaling statistics
print(f"\nScaling verification (mean ≈ 0, std ≈ 1):")
scaling_stats = df_scaled[numerical_features].agg(['mean', 'std']).round(3)
print(scaling_stats.head())

## Step 6: Feature Selection and Final Dataset

In [ ]:
print("=== FEATURE SELECTION ===")

# Remove original categorical columns and redundant features
columns_to_drop = [
    'income',  # Original target (use income_target)
    'fnlwgt',  # Not relevant for prediction
    'education-num',  # Redundant with education_rank
]

# Add original categorical columns that were encoded
original_categorical = ['workclass', 'education', 'marital-status', 'occupation', 
                       'relationship', 'race', 'sex', 'native-country']
columns_to_drop.extend([col for col in original_categorical if col in df_scaled.columns])

# Remove columns
df_final = df_scaled.drop(columns=[col for col in columns_to_drop if col in df_scaled.columns])

print(f"Columns removed: {[col for col in columns_to_drop if col in df_scaled.columns]}")
print(f"Final dataset shape: {df_final.shape}")

# Separate features and target
X = df_final.drop('income_target', axis=1)
y = df_final['income_target']

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nFeature columns ({len(X.columns)}):")
feature_columns = X.columns.tolist()
for i, col in enumerate(feature_columns[:20]):  # Show first 20
    print(f"  {i+1:2d}. {col}")
if len(feature_columns) > 20:
    print(f"  ... and {len(feature_columns) - 20} more columns")

## Step 7: Train-Test Split

In [ ]:
print("=== TRAIN-TEST SPLIT ===")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape} features, {y_train.shape} targets")
print(f"Test set: {X_test.shape} features, {y_test.shape} targets")

# Verify stratification
print(f"\nTarget distribution verification:")
train_dist = y_train.value_counts(normalize=True) * 100
test_dist = y_test.value_counts(normalize=True) * 100

print(f"Training set distribution:")
for value, pct in train_dist.items():
    label = target_encoder.inverse_transform([value])[0]
    print(f"  {label}: {pct:.1f}%")

print(f"Test set distribution:")
for value, pct in test_dist.items():
    label = target_encoder.inverse_transform([value])[0]
    print(f"  {label}: {pct:.1f}%")

## Step 8: Save Processed Data

In [ ]:
print("=== SAVING PROCESSED DATA ===")

# Save processed datasets
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False, header=['income_target'])
y_test.to_csv('../data/y_test.csv', index=False, header=['income_target'])

# Save complete processed dataset
df_final.to_csv('../data/data_processed.csv', index=False)

print("✓ Processed datasets saved:")
print("  - X_train.csv")
print("  - X_test.csv")
print("  - y_train.csv")
print("  - y_test.csv")
print("  - data_processed.csv")

# Save encoders for future use
import joblib

joblib.dump(scaler, '../data/scaler.pkl')
joblib.dump(target_encoder, '../data/target_encoder.pkl')
joblib.dump(label_encoders, '../data/label_encoders.pkl')

print("\n✓ Encoders saved:")
print("  - scaler.pkl")
print("  - target_encoder.pkl")
print("  - label_encoders.pkl")

## Step 9: Feature Engineering Summary

In [ ]:
print("=" * 80)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 80)

print(f"\n📊 DATASET TRANSFORMATION:")
print(f"   • Original shape: {df_original.shape}")
print(f"   • Final shape: {df_final.shape}")
print(f"   • Features created: {df_final.shape[1] - df_original.shape[1] + 1}")
print(f"   • Training samples: {X_train.shape[0]:,}")
print(f"   • Test samples: {X_test.shape[0]:,}")

print(f"\n🛠️ FEATURE ENGINEERING OPERATIONS:")
print(f"   ✓ Created derived features (education rank, age groups, work intensity)")
print(f"   ✓ Created binary indicators (overtime, capital flags, marriage status)")
print(f"   ✓ Created interaction features (age×education, hours×education, etc.)")
print(f"   ✓ Encoded categorical variables (label + one-hot encoding)")
print(f"   ✓ Scaled numerical features (StandardScaler)")
print(f"   ✓ Prepared train-test split (80-20, stratified)")

print(f"\n🎯 KEY FEATURES CREATED:")
key_features = ['education_rank', 'overtime', 'is_married', 'is_us_native',
               'has_capital_gain', 'age_education_interaction']
for feature in key_features:
    if feature in X.columns:
        print(f"   • {feature}")

print(f"\n📋 ENCODING SUMMARY:")
encoded_features = [col for col in X.columns if '_' in col and any(x in col for x in ['workclass', 'education', 'marital', 'occupation'])]
print(f"   • One-hot encoded features: {len(encoded_features)}")
print(f"   • Label encoded features: {len(binary_cols)}")
print(f"   • Scaled numerical features: {len(numerical_features)}")

print(f"\n🎲 DATA READY FOR MODELING:")
print(f"   • Balanced classes: {(y_train.value_counts() / len(y_train) * 100).round(1).to_dict()}")
print(f"   • No missing values: {X_train.isnull().sum().sum() == 0}")
print(f"   • Features standardized: ✓")
print(f"   • Ready for ML algorithms: ✓")

print("\n" + "=" * 80)

## Summary and Next Steps

**Feature Engineering Completed Successfully! 🎉**

**Key Accomplishments:**
- ✅ **Created 10+ new derived features** (education rank, work intensity, capital flags)
- ✅ **Built interaction features** (age×education, hours×education combinations)
- ✅ **Encoded categorical variables** (one-hot for nominal, label for binary)
- ✅ **Scaled numerical features** (StandardScaler for mean=0, std=1)
- ✅ **Prepared modeling datasets** (stratified train-test split)
- ✅ **Saved all preprocessors** (scalers, encoders for future use)

**Dataset Ready for Modeling:**
- 📈 **Training set**: 36,155 samples × 50+ features
- 📊 **Test set**: 9,039 samples × 50+ features
- ⚖️ **Balanced classes** maintained in split
- 🔢 **All features properly encoded and scaled**

**Key Features for Prediction:**
- `education_rank` (1-16 education hierarchy)
- `age`, `hours-per-week` (scaled)
- `overtime`, `is_married`, `is_us_native` (binary flags)
- `age_education_interaction` (experience × education)
- One-hot encoded: occupation, workclass, marital-status

**Next Steps:**
1. ➡️ **05_model_training.ipynb**: Train multiple ML algorithms
2. **06_model_evaluation.ipynb**: Evaluate and compare model performance
3. **07_model_deployment.ipynb**: Deploy best model for predictions